# 0.4 Propagation of a known scalar optical field

[0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb) prescribed a field just after a slit. Here we follow its spreading through a homogeneous medium: identify wave components, propagate them and recombine them into the field on another plane.

## Motivation and prerequisites

The needed ideas from Notebooks [0.1](0_1_Scalar_Fields_and_Helmholtz.ipynb)–[0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb) are amplitude and phase, the Helmholtz equation, superposition and the distinction between a field prescription and a physical scattering problem. No Fourier optics or FFT knowledge is assumed.

Fourier transforms convert transverse derivatives into multiplication; ordinary differential equations give each component's propagation factor. The normalization and branch-choice derivations can be studied after the plots.

Watch the slit field spread, connect narrow spatial features with a broad spectrum, then follow one component. Use those observations to interpret the angular spectrum method (ASM). The 0 series is a conceptual overview of scalar diffraction physics; subsequent numbered notebooks revisit its concepts and mathematics systematically.

[0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) supplies the missing input by determining the complex field from material scattering. [0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb) decomposes and propagates that scattered field; [0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb) compares it with the ideal binary-mask approximation.

Run this notebook independently from top to bottom. Each explorer has its own controls and reset button. We use $e^{-i\omega t}$, wavelength $\lambda$ **in the medium**, and propagation toward **$+z$**, as in [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb). The physical grating lessons use transmission toward $z<0$.

[Previous: 0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb) · [Chapter index](0_0_Foundations_of_Scalar_Optics.ipynb)


:::{note} Interactive examples
Click the **power button** near the top of the page to activate the examples. Binder supplies Python in your browser; no local installation is needed. Wait for the session to connect and the code to finish running. Then try **Keep spectrum phase**, **Transverse frequency**, and **Observation z** in their respective explorers. Open the optional **ASM versus Fresnel** experiment to compare propagation factors. Use **Reset this example** to restore each starting view. The text, equations and saved figures can be read without activating a session.
:::

For local use, download the {download}`numerical explorer <downloads/grating_physics_numerical_explorer.zip>` or the complete {download}`worked-example scripts <downloads/grating_physics_scripts.zip>` with their shared helpers.

In [ ]:
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import ipywidgets as widgets
from IPython.display import display, Image
from wave_optics.scalar_propagation_view import ScalarPropagationExplorer, slit_setup_png
from wave_optics.grating_intro import _close_widget_tree

# Release the previous widget trees when this setup cell is rerun.
if 'propagation_lessons' in globals():
    for old_lesson in propagation_lessons.values():
        old_lesson.close()
if 'propagation_panels' in globals():
    for old_panel in propagation_panels.values():
        _close_widget_tree(old_panel)
propagation_lessons, propagation_panels = {}, {}

def show_propagation(stage, optional=False):
    if stage in propagation_lessons:
        propagation_lessons[stage].close()
    if stage in propagation_panels:
        _close_widget_tree(propagation_panels.pop(stage))
    lesson = ScalarPropagationExplorer(stage)
    propagation_lessons[stage] = lesson
    if optional:
        panel = widgets.Accordion(children=[lesson.ui], selected_index=None)
        panel.set_title(0, 'Optional experiment: ASM versus Fresnel')
        propagation_panels[stage] = panel
        display(panel)
    else:
        display(lesson.ui)


## 1. We know the field. What happens next?

$$\boxed{U_{\mathrm{exit}}(x,y)=U(x,y,0^+).}$$

A normally incident plane wave illuminates the ideal long slit from [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb). Immediately after the thin mask, we prescribe unit amplitude and zero phase inside the opening, and zero field outside. The slit is uniform along $y$, so the picture needs only $x$ and $z$.

**Predict:** will the region beyond the blocked part stay dark? Observe the intensity at the right: why does light spread beyond the geometrical opening even in a homogeneous medium?

In [ ]:
display(Image(data=slit_setup_png()))


The figure uses $W_{\mathrm{open}}=20\,\mu\mathrm m$ and $\lambda=0.6\,\mu\mathrm m$; incident crest spacing is schematic. The observed field extends outside the slit’s geometrical shadow. Its propagation must account for **amplitude and phase**, not just trace rays through the opening.

Our input is the prescribed thin-mask field $U_{\mathrm{exit}}=tU_i$. The angular spectrum method (ASM) accepts this field; it does not determine the physical aperture’s transmission. We also specify **no incoming wave from the distant $+z$ side**. Together, the known field and this outgoing condition select the propagation solution.

Knowing intensity alone is insufficient: $U_{\mathrm{exit}}$ and $U_{\mathrm{exit}}e^{i\phi(x,y)}$ have the same boundary intensity but can produce different patterns after propagation.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## 2. A complicated field can be built from plane waves

**Question:** how can the plane waves of [0.2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb) represent a field confined to a slit?

Each component extends across the whole plane. Their complex contributions add inside the opening and cancel outside it. A **spatial frequency** counts phase cycles per unit distance across the plane; it is not a new optical frequency.

**Predict:** does a narrower opening require a smaller or larger range of spatial frequencies?

**Explore:** change the slit width from 20 to 10 µm. Then uncheck **Keep spectrum phase**. Can the same spectrum magnitude still reconstruct the aperture? Restore the checkbox before continuing.

In [ ]:
show_propagation('spectrum')


A narrow feature needs finer spatial variation to build its edges, so its spectrum is broader. The dashed markers locate the first spectral zeros. Removing phase spoils the cancellation even though the magnitude plot stays the same.

The **angular spectrum** $\widetilde U_{\mathrm{exit}}$ gives the complex weight of each component:

$$\boxed{U_{\mathrm{exit}}(x,y)=\iint \widetilde U_{\mathrm{exit}}(f_x,f_y)e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y,\qquad
k_x=2\pi f_x,\quad k_y=2\pi f_y.}$$

The integral means “add all the components.” For our $y$-invariant slit, only $f_y=0$ is needed, so the computation is one-dimensional. The displayed magnitude is only one part of $\widetilde U_{\mathrm{exit}}$; reconstruction uses its phase too.

### Why do we choose plane waves?

Start with a plane wave with a real wavevector,

$$U_{\mathbf k}(\mathbf r)=e^{i\mathbf k\cdot\mathbf r},\qquad
\boxed{\nabla^2e^{i\mathbf k\cdot\mathbf r}=-|\mathbf k|^2e^{i\mathbf k\cdot\mathbf r}.}$$

The Laplacian changes only a multiplicative factor, leaving the function's form intact. Such a function is called an **eigenfunction** of the operator. For this wave to satisfy Helmholtz, $|\mathbf k|=k$.

In real space, a field may have a complicated amplitude and phase distribution. In Fourier space, it becomes a collection of components with definite transverse spatial frequencies and **complex spectral amplitudes**. The angular spectrum contains phase as well as magnitude; it is not a spectral intensity image.

These components are convenient building blocks because spatial derivatives act on each separately. In a homogeneous medium, we can evolve each component independently and then add them to reconstruct the field. This is why plane waves appear throughout the grating notebooks. Other useful solution families exist, as [0.2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb) showed.

A transverse Fourier component does not always extend into a propagating wave with a real wavevector. Section 3 shows when a complex $k_z$ is required, producing an evanescent component.

:::{dropdown} Optional mathematics: the transform pair and slit spectrum

With the forward-transform convention

$$\widetilde U_{\mathrm{exit}}(f_x,f_y)=\iint U_{\mathrm{exit}}(x,y)e^{-i2\pi(f_xx+f_yy)}\,dx\,dy,$$

the inverse transform is the sum above. For a unit field in a centred slit of full width $W_{\mathrm{open}}$, the one-dimensional transform is

$$\widetilde U_{\mathrm{exit}}(f_x)=\int_{-W_{\mathrm{open}}/2}^{W_{\mathrm{open}}/2}e^{-i2\pi f_xx}\,dx
=W_{\mathrm{open}}\,\operatorname{sinc}(W_{\mathrm{open}}f_x),\qquad
\operatorname{sinc}(q)=\frac{\sin(\pi q)}{\pi q}.$$

Its first zeros lie at $f_x=\pm1/W_{\mathrm{open}}$. Negative lobes represent phase $\pi$, which is lost if we replace $\widetilde U_{\mathrm{exit}}$ with $|\widetilde U_{\mathrm{exit}}|$. For this symmetric example the reconstructed field remains real, but it is no longer the slit field.

:::

### Checkpoint: can magnitude alone rebuild the slit?

**Predict:** will discarding the spectrum's phase still recover the opening?

**Try it in code:** run this cell, then change `try_keep_phase` to `False`. It uses the sampled slit from the explorer above. The FFT and inverse FFT use a matching discrete normalization.


In [ ]:
import numpy as np

try_keep_phase = True  # Change to False, then run this cell again.
try_u0 = propagation_lessons['spectrum'].data['boundary_field']
try_spectrum = np.fft.fft(try_u0)
try_weights = try_spectrum if try_keep_phase else np.abs(try_spectrum)
try_rebuilt = np.fft.ifft(try_weights)
print(f"Maximum field mismatch: {np.max(np.abs(try_rebuilt - try_u0)):.3e}")


:::{dropdown} Check your prediction


Keeping the full complex spectrum recovers the sampled field to rounding precision. Keeping only its magnitude changes the cancellation between components, so the reconstructed field differs. This is a decomposition check; it does not propagate the field.


:::


## 3. How does each component propagate?

**Question:** once a component’s transverse variation is known, what determines its variation along $z$?

The Helmholtz relation from [0.2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb) is

$$\boxed{k_x^2+k_y^2+k_z^2=k^2,\qquad k=\frac{2\pi}{\lambda}.}$$

### Why does Fourier decomposition simplify the Helmholtz equation?

The homogeneous scalar Helmholtz equation is a **partial differential equation (PDE)** involving three coordinates:

$$\frac{\partial^2U}{\partial x^2}+\frac{\partial^2U}{\partial y^2}
+\frac{\partial^2U}{\partial z^2}+k^2U=0.$$

Transform only $x,y$; keep $z$ as the propagation coordinate. For this derivation, write the spectrum in wavevector coordinates as $A_k$, with $k_x=2\pi f_x$ and $k_y=2\pi f_y$:

$$A_k(k_x,k_y;z)=\iint U(x,y,z)e^{-i(k_xx+k_yy)}\,dx\,dy.$$

Fourier transformation turns the transverse derivatives into multiplication:

$$\frac{\partial^2U}{\partial x^2}\longrightarrow-k_x^2A_k,\qquad
\frac{\partial^2U}{\partial y^2}\longrightarrow-k_y^2A_k.$$

For each fixed pair $(k_x,k_y)$, only $z$ remains variable:

$$\boxed{\frac{d^2A_k}{dz^2}+(k^2-k_x^2-k_y^2)A_k=0,\qquad
k_z^2=k^2-k_x^2-k_y^2.}$$

**One PDE becomes a family of independent ordinary differential equations (ODEs), one for every transverse wavevector pair.** Each equation tells us how that component changes along $z$.

**Helmholtz PDE → transform $x,y$ → independent ODEs in $z$ → solve → reconstruct.**

The decomposition supplies $k_x,k_y$; Helmholtz supplies the allowed $k_z$.

**Predict:** if $k_x^2+k_y^2$ exceeds $k^2$, can $k_z$ still be real?

**Explore:** the plot follows a unit component with $f_y=0$. Move $\lambda f_x$ from 0.60 to 1.00, then to 1.25. Watch the dashed amplitude as well as the signed field factor.

In [ ]:
show_propagation('component')


Solving the ODE gives, for $k_z\ne0$,

$$A_k(k_x,k_y;z)=C_+(k_x,k_y)e^{ik_zz}+C_-(k_x,k_y)e^{-ik_zz}.$$

With our $e^{-i\omega t}$ convention, choose the **outgoing or decaying** $k_z$ branch below. No incoming wave from the distant $+z$ side, and no exponentially growing field there, select $C_-=0$. The known input sets $C_+=A_k(k_x,k_y;0)$, giving

$$\boxed{A_k(k_x,k_y;z)=A_k(k_x,k_y;0)\,H_{\rm ASM},\qquad H_{\rm ASM}=e^{ik_zz}.}$$

**The ASM factor follows from solving Helmholtz for each component.** The plot shows the resulting phase advance or decay:


| Transverse content | Longitudinal component | What changes with $z$? |
| --- | --- | --- |
| $k_x^2+k_y^2<k^2$ | $k_z=+\sqrt{k^2-k_x^2-k_y^2}$ | Phase advances; amplitude stays fixed. |
| $k_x^2+k_y^2=k^2$ | $k_z=0$ | Grazing component: no $z$ variation. |
| $k_x^2+k_y^2>k^2$ | $k_z=i\kappa$, $\kappa>0$ | $H=e^{-\kappa z}$: amplitude decays. |

At exact grazing, $k_z=0$ gives $\frac{d^2A_k}{dz^2}=0$. Boundedness selects its constant solution, so $H=1$.

An evanescent component has transverse variation too rapid to correspond to a real propagation direction at this wavelength. It can describe structure near the prescribed plane, but decays away from it. [Further reading: TU Delft, angular-spectrum propagation](#ref-0-4-delft-wavefield) [\[1\]](#ref-0-4-delft-wavefield).

### When do different components couple?

Here $k$ is spatially constant: the transform of $k^2U$ is simply $k^2A_k$. No other transverse frequency enters a component's ODE.

In a scalar model with a transversely varying refractive index, instead we have

$$\nabla^2U+k_0^2n^2(x,y,z)U=0,\qquad k_0=\frac{2\pi}{\lambda_0},$$

where $\lambda_0$ is the vacuum wavelength. The transform of the product $n^2U$ is a **convolution**: it combines the index distribution's spectrum with the field's spectrum, generally coupling different transverse frequencies.

Thus homogeneous propagation evolves the components independently, while a structured material can scatter field into other components. The simple ASM multiplier applies in the homogeneous output region; a later scattering model must determine what the material produces. Variation only along $z$ would still leave the transverse components independent, although their propagation equations would have variable coefficients.

:::{dropdown} Optional mathematics: derivatives, normalization, and branch choices

For a sufficiently decaying field, integrating by parts twice gives

$$\begin{aligned}
\mathcal F_{xy}\!\left\{\frac{\partial U}{\partial x}\right\}&=ik_xA_k,\\
\mathcal F_{xy}\!\left\{\frac{\partial^2U}{\partial x^2}\right\}&=(ik_x)^2A_k=-k_x^2A_k.
\end{aligned}$$

The boundary terms vanish; the same result holds for $y$. Since the transform does not act on $z$, $\mathcal F_{xy}\{\partial_z^2U\}=\partial_z^2A_k$. Substituting these three results gives the ODE above. Ideal plane waves and sharp prescribed apertures can be treated using generalized Fourier transforms.

**Normalization:** let $\widetilde U_{\mathrm{exit}}(f_x,f_y;z)$ denote the frequency-coordinate spectrum at any plane, extending Section 2's notation. Then

$$A_k(k_x,k_y;z)=\widetilde U_{\mathrm{exit}}\!\left(\frac{k_x}{2\pi},\frac{k_y}{2\pi};z\right),$$

but the inverse integral in wavevector coordinates carries a factor:

$$U(x,y,z)=\frac{1}{(2\pi)^2}\iint A_k(k_x,k_y;z)e^{i(k_xx+k_yy)}\,dk_x\,dk_y.$$

This follows from $dk_x\,dk_y=(2\pi)^2df_x\,df_y$. It leaves the notebook's frequency-coordinate convention and FFT normalization unchanged.

**Branches:** take $\operatorname{Re}k_z\geq0$ and $\operatorname{Im}k_z\geq0$. For real positive $k_z$, $e^{ik_zz-i\omega t}$ travels toward $+z$. For $k_z=i\kappa$, $\kappa>0$, the two solutions are $e^{-\kappa z}$ and $e^{+\kappa z}$; boundedness selects decay. At grazing, $A_k=C_0+D_0z$, and boundedness requires $D_0=0$.

For a complex wavevector, the Laplacian eigenvalue is $-(k_x^2+k_y^2+k_z^2)$, without complex conjugation; the real-vector expression $-|\mathbf k|^2$ from Section 2 must not be used as a Hermitian norm here. This lesson propagates only to $z\geq0$; reversing the evanescent factor would exponentially amplify errors.

:::

### Checkpoint: phase advance or amplitude decay?

**Predict:** what happens to a component's amplitude after one wavelength of propagation?

**Try it in code:** change `try_alpha` from `0.60` to `1.00` and `1.25`, then double `try_distance`. Here $\alpha=\lambda f_x$ and $f_y=0$, matching the component explorer.


In [ ]:
from wave_optics.scalar_propagation import transfer_function

try_wavelength = 0.6  # Micrometres in the medium.
try_alpha = 0.60
try_distance = try_wavelength  # Positive distance along +z, in micrometres.
try_H = transfer_function(try_alpha / try_wavelength, 0.0,
                          try_wavelength, try_distance)
print(f"H = {complex(try_H):+.6f}")
print(f"Amplitude multiplier |H| = {abs(try_H):.6f}")


:::{dropdown} Check your prediction


For $\alpha=0.60$, $|H|=1$: only phase changes. At $\alpha=1$, $H=1$ at every distance. At $\alpha=1.25$, $|H|=e^{-1.5\pi}\approx0.00898$ after one wavelength; doubling the distance squares that amplitude multiplier. This decay is in a homogeneous lossless medium.


:::


## 4. Reconstruct the propagated field

**Question:** what happens when all those independently evolving components are added again?

**Predict:** should the observation plane near the slit resemble the opening more closely than a distant plane?

**Explore:** leave the width at 20 µm and move **Observation z** from 0 to 20, 200 and 1000 µm. Then use width 10 µm and wavelength 0.8 µm at 1000 µm to approach the broad far-field shape. Compare with width 40 µm at the same wavelength and distance.

In [ ]:
show_propagation('slit')


The cyan line selects a slice of the same calculated field. Near the slit, its profile retains edge structure; farther away, the contributions form a broader diffraction pattern. Narrowing the slit broadens the angular content. Increasing wavelength gives a larger angle for the same propagating spatial frequency, since $\sin\theta=\lambda|f_x|$ when $f_y=0$.

ASM performs three steps:

**Decompose the input → propagate each component → add the components again.**

$$\boxed{U(x,y,z)=\mathcal F^{-1}\!\left[\mathcal F\{U_{\mathrm{exit}}(x,y)\}\,e^{ik_zz}\right].}$$

Here $\mathcal F$ is the Fourier decomposition and $\mathcal F^{-1}$ reconstructs the field. At $z=0$, every multiplier is 1, so reconstruction gives the original field. Away from that plane, different components acquire different phase changes (or decay), changing their sum.

The plots show **cycle-averaged relative intensity** $I_{\rm rel}=|U|^2$, using unit incident intensity throughout. They do not show the instantaneous signed field $u=\operatorname{Re}\{Ue^{-i\omega t}\}$. A zero of $u$ at one instant need not be a dark point. The colour scale is fixed; darker distant profiles are not rescaled to look equally bright.

:::{dropdown} Optional numerical details: sampling, padding and FFT coefficients

An FFT is a fast way to calculate a **discrete** Fourier sum. It approximates the continuous transform on a sampled, finite, periodically repeated domain. The large empty region around our slit separates its numerical copies; it is not another physical aperture.

The propagation uses a 4096 µm window, spacing $\Delta x=0.0625$ µm, and 65,536 samples. It displays only $x=-80$ to ${80}$ µm at 0.5 µm intervals and $z=0$ to ${1000}$ µm at 5 µm intervals. Each detector curve uses the same reference as the map. The edge cell uses its average transmission. The view illustrates diffraction over these scales; it does not resolve every sub-wavelength feature at the aperture edge.

With $N$ spatial samples, $\Delta f_x=1/(N\Delta x)$ and the Nyquist limit is $1/(2\Delta x)$. The slit’s sharp edge has unlimited spectral content, so any sampled representation is approximate. Finer spatial sampling resolves more content; a wider padded window improves frequency sampling and separates periodic copies. Padding alone cannot fix unresolved edges.

For a centred one-dimensional grid, the continuous spectrum is approximated by

$$\widetilde U_{\mathrm{exit}}(f_m)\simeq\Delta x\sum_j U_{\mathrm{exit}}(x_j)e^{-i2\pi f_mx_j}.$$

Raw FFT coefficients omit $\Delta x$ (or $\Delta x\Delta y$ in two dimensions). The spectrum illustration includes this factor and the centred-origin shifts; its inverse undoes them. The propagation calculation uses a matched FFT/inverse-FFT pair, so their normalization factors cancel. The spectrum illustration uses a smaller 512 µm window because it only decomposes and reconstructs the input, without propagation.

Regression checks halve $\Delta x$ and double the propagation window at the control-range corners. At the checked planes, absolute intensity differences stay below 0.004 and 0.006 respectively, relative to incident intensity 1. These are checks of this displayed example, not universal accuracy guarantees. Do not extrapolate the controls to arbitrary distances without checking sampling and wraparound again.

:::

## 5. How does ASM relate to other methods?

**Question:** would a different propagation formula describe different diffraction physics?

**Predict:** if two formulations solve the same scalar problem without approximation, should their fields agree?

| Method | Central idea |
| --- | --- |
| **ASM** | Propagate plane-wave components in spatial-frequency space. |
| **Rayleigh–Sommerfeld** | Propagate the boundary field with a spatial-domain diffraction kernel. |
| **Fresnel** | Use a paraxial approximation, retaining quadratic phase. |
| **Fraunhofer** | Use the far-field limit; the pattern follows the aperture’s spectrum. |

ASM and the corresponding Rayleigh–Sommerfeld formulation express the same outgoing scalar propagation problem. Fresnel and Fraunhofer simplify it under additional conditions. We use ASM as the numerical reference. [Further reading: TU Delft, scalar diffraction optics](#ref-0-4-delft-diffraction) [\[2\]](#ref-0-4-delft-diffraction).

For the slit, the full-width parameter $W_{\mathrm{open}}^2/(\lambda z)$ helps distinguish near and far behavior: values much smaller than 1 support the paraxial far-field approximation. At $W_{\mathrm{open}}=10$ µm, $\lambda=0.8$ µm and $z=1000$ µm it is 0.125, and the central profile already approaches the sinc-squared shape from [0.3](0_3_Boundary_Conditions_and_Diffraction.ipynb). This is a gradual approximation, not a sharp boundary.

:::{dropdown} Optional explanation for the comparison below

Let $s=\lambda\sqrt{f_x^2+f_y^2}$. For small $s$,

$$k_z=k\sqrt{1-s^2}\simeq k\left(1-\frac{s^2}{2}\right),\qquad
H_F=e^{ikz}\,e^{-i\pi\lambda z(f_x^2+f_y^2)}.$$

**Try:** increase $z/\lambda$ from 2 to 10. Compare the small-$s$ part with the region near $s=1$. The right panel shows the accumulated, unwrapped ASM phase minus Fresnel phase. Its leading value is $-kzs^4/8$, so a small error per wavelength can accumulate with distance. Fresnel also lacks evanescent decay and should not be used for that part of the spectrum. Wavelength rescales physical units while these normalized curves stay fixed.

:::

In [ ]:
show_propagation('transfer', optional=True)


## 6. What have we learned?

**Known complex field → plane-wave components → Helmholtz determines $k_z$ → propagate → reconstruct.**

The wave equation permits many fields; the prescribed boundary field and outgoing condition select this one. ASM extends that field through a homogeneous medium. Finding the true field inside or immediately outside a physical aperture or grating remains a separate scattering problem.

**Check your understanding:** why is an intensity image alone insufficient as ASM input? Why does a narrow slit need a broad spectrum? Which components change amplitude during propagation?

:::{dropdown} Check your answers

Intensity omits phase, which controls cancellation and reinforcement. Narrow features require a wider range of transverse phase variations. Propagating components retain their amplitudes; evanescent components decay. Their recombined field can still change amplitude from point to point and plane to plane.

:::

**Connection to diffraction orders:** Notebook [0.2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb) introduced plane waves as Helmholtz solutions; [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb) explains why they form a natural propagation basis. Later grating lessons show how an ideal infinite periodic grating under plane-wave illumination restricts the transverse wavevectors to discrete **Floquet harmonics**. For period $d$ along $x$,

$$k_{x,m}=k_{x,i}+mG,\qquad G=\frac{2\pi}{d},\qquad m\in\mathbb Z,$$

where $k_{x,i}$ is the incident transverse wavevector component. In the homogeneous output region, reconstruct the field by summing these harmonics:

$$\boxed{U(\mathbf r)=\sum_m C_m e^{i\mathbf k_m\cdot\mathbf r}.}$$

Each harmonic uses the appropriate outgoing or decaying longitudinal branch for the chosen output half-space. Its complex coefficient $C_m$ depends on the prescribed field or scattering model; propagation alone does not determine it from material geometry. Next, [0.5: Physical gratings and the Helmholtz boundary-value problem](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) determines the field from a finite-thickness, transversely periodic absorbing structure. [0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb) then decomposes and propagates that field, and [0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb) compares it with an ideal binary amplitude mask.
- [Chapter 1](1_0_Plane_Waves_and_Wavevectors.ipynb) subsequently returns to plane-wave and wavevector geometry for detailed development. 

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete {download}`scripts download <downloads/grating_physics_scripts.zip>` to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. A complicated field can be built from plane waves

**Script:** [0_4_01_fft_reconstruction_phase.py](Wave_solver_py_scripts/scripts/chapter_0/0_4_01_fft_reconstruction_phase.py)

Reconstruct a complex field with and without spectral phase.

### 2. How does each component propagate?

**Script:** [0_4_02_component_multiplier.py](Wave_solver_py_scripts/scripts/chapter_0/0_4_02_component_multiplier.py)

Choose the outgoing square-root branch for each component.

### 3. Reconstruct the propagated field

**Script:** [0_4_03_asm_slit_from_scratch.py](Wave_solver_py_scripts/scripts/chapter_0/0_4_03_asm_slit_from_scratch.py)

Propagate a sampled slit by the angular-spectrum method.

### 4. How does ASM relate to numerical sampling?

**Script:** [0_4_04_sampling_and_padding.py](Wave_solver_py_scripts/scripts/chapter_0/0_4_04_sampling_and_padding.py)

Separate FFT resolution from window-padding effects.




(refs-0-4)=

## References

(ref-0-4-delft-wavefield)=

[1] TU Delft, “Wavefield propagation,” *Advanced Optical Imaging*. [Online]. Available: [source](https://qiweb.tudelft.nl/aoi/wavefieldpropagation/wavefieldpropagation/).

**Reading notes:** §3.6 (Angular spectrum). Connects transverse Fourier components with longitudinal propagation and evanescent decay; useful alongside Sections 2–3.

(ref-0-4-delft-diffraction)=

[2] A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, “Scalar Diffraction Optics,” TU Delft. [Online]. Available: [source](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html).

**Reading notes:** Read “Angular Spectrum Method,” “Rayleigh-Sommerfeld Diffraction Integral” and the Fresnel/Fraunhofer discussion to compare the methods in Section 5.

## Further Reading

(ref-0-4-numpy-126)=

- NumPy developers, “Discrete Fourier Transform,” *NumPy 1.26 Manual*. [Online]. Available: [source](https://numpy.org/doc/1.26/reference/routines.fft.html).

  **Reading notes:** A coding reference for the optional snippet: distinguish discrete FFT normalization from the continuous integral's sample-spacing factor.

(ref-0-4-bandlimited)=

- K. Matsushima and T. Shimobaba, “Band-Limited Angular Spectrum Method for Numerical Simulation of Free-Space Propagation in Far and Near Fields,” *Optics Express*, vol. 17, pp. 19662–19673, 2009. [Online]. Available: [PDF](https://kansai-u.repo.nii.ac.jp/record/11127/files/KU-1100-20091015-50.pdf). doi: 10.1364/OE.17.019662.

  **Reading notes:** **Access:** Free institutional-repository full text; copyright retained. **Read for:** Read the sampling discussion; this notebook does not implement that algorithm.

(ref-0-4-mansuripur)=

- M. Mansuripur, “A Tutorial on the Classical Theories of Electromagnetic Scattering and Diffraction,” *Nanophotonics*, vol. 10, pp. 315–342, 2021; arXiv version submitted 2020. [Online]. Available: [preprint](https://arxiv.org/abs/2009.06706).

  **Reading notes:** **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional advanced reading: scalar diffraction and its electromagnetic assumptions.

## Reference notes

Start with the physical explanations; the derivations are optional.

Freely accessible works are not necessarily in the public domain.




## Continue into systematic propagation

[8.0: Introduction to field propagation](8_0_Introduction_to_Field_Propagation.ipynb) opens Chapter 8 after 7.4. It builds on this early preview with known grating-order coefficients and the later outgoing negative-z convention. 8.0 introduces the chapter; [8.1](8_1_Physical_Pictures_Behind_Scalar_Field_Propagation.ipynb) compares the physical pictures. Lessons 8.2-8.5 are planned.
